In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import torch
from torchvision import transforms
from torchmetrics.functional.classification import multiclass_f1_score
from pixel_arena.dataset_utils.celeb_a_mask_hq import LABELS

In [ ]:
image_id = "3b556b6f99974aa490b57b9e3aaddcb6"


original_image_path = Path("./eval-set/celeb/images-150") / f"{image_id}.jpg"
reference_mask_path = Path("./eval-set/celeb/masks-1024") / f"{image_id}.png"
reference_mask_path_512 = Path("./eval-set/celeb/masks-512") / f"{image_id}.png"
gemini_pro_result_path = (
    Path("./results/celeb/gemini-pro-150") / f"{image_id}.mask.0.pred.png"
)
# gemini_result_path = Path("./results/celeb/gemini-150") / f"{image_id}.mask.0.pred.png"
# gpt_result_path = Path("./results/celeb/gpt-image-150") / f"{image_id}.mask.0.pred.png"
# sam3_result_path = Path("./results/celeb/sam3-150") / f"{image_id}.mask.0.pred.png"
segface_result_path = Path("./results/celeb/segface-150") / f"{image_id}.mask.0.pred.png"

In [ ]:
# Compute F1 scores
mask_transform = transforms.PILToTensor()
label_num = len(LABELS)

# Load masks as tensors
reference_mask = Image.open(reference_mask_path)
reference_mask_512 = Image.open(reference_mask_path_512)
reference_mask_tensor = mask_transform(reference_mask)
reference_mask_512_tensor = mask_transform(reference_mask_512)

gemini_pro_mask = Image.open(gemini_pro_result_path)
gemini_pro_mask_tensor = mask_transform(gemini_pro_mask)

segface_mask = Image.open(segface_result_path)
segface_mask_tensor = mask_transform(segface_mask)

# Compute F1 scores
gemini_pro_f1 = multiclass_f1_score(
    gemini_pro_mask_tensor,
    reference_mask_tensor,
    num_classes=label_num,
    average="macro",
)

segface_f1 = multiclass_f1_score(
    segface_mask_tensor,
    reference_mask_512_tensor,
    num_classes=label_num,
    average="macro",
)

print(f"Gemini Pro F1 Score: {gemini_pro_f1:.4f}")
print(f"SegFace F1 Score: {segface_f1:.4f}")

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10, 10))

# Load images
original_image = Image.open(original_image_path)
reference_mask = Image.open(reference_mask_path)
gemini_pro_image = Image.open(gemini_pro_result_path)
segface_image = Image.open(segface_result_path)

# Top row
axs[0, 0].imshow(original_image)
axs[0, 0].set_title("Original Image")
axs[0, 0].axis("off")

axs[0, 1].imshow(reference_mask)
axs[0, 1].set_title("Reference Mask")
axs[0, 1].axis("off")

# Bottom row
axs[1, 0].imshow(segface_image)
axs[1, 0].set_title(f"segface: f1={segface_f1:.4f}")
axs[1, 0].axis("off")

axs[1, 1].imshow(gemini_pro_image)
axs[1, 1].set_title(f"gemini-3-pro-image-preview: f1={gemini_pro_f1:.4f}")
axs[1, 1].axis("off")

plt.tight_layout()
plt.show()